In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from ml.preprocess import audio_to_mel, build_label_map, encode_labels, save_label_map
from ml.model import build_model

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
if DEVICE.type == 'mps':
    torch.backends.mps.enable_fallback_kernels = True

NUM_CLASSES = 20
SAVED_MODELS = Path('../saved_models')
TRAINING_LOGS = Path('../training_logs')
SAVED_MODELS.mkdir(exist_ok=True)
TRAINING_LOGS.mkdir(exist_ok=True)

print(f'Device: {DEVICE}')

## Загрузка данных

In [ ]:
train_x = np.load('../Data/train_x.npy', allow_pickle=True)
train_y = np.load('../Data/train_y.npy', allow_pickle=True)
valid_x = np.load('../Data/valid_x.npy', allow_pickle=True)
valid_y = np.load('../Data/valid_y.npy', allow_pickle=True)

print(f'Train: {train_x.shape}, Valid: {valid_x.shape}')

## Предвычисление спектрограмм (один раз для всех trials)

In [ ]:
def precompute_spectrograms(X: np.ndarray, Y: np.ndarray, n_mels: int, label_map: dict) -> tuple:
    mels = [audio_to_mel(sig, n_mels=n_mels) for sig in X]
    X_t = torch.stack(mels)
    y_t = torch.tensor(encode_labels(Y, label_map), dtype=torch.long)
    return X_t, y_t


N_MELS_VARIANTS = [32, 64, 128]

label_map = build_label_map(np.concatenate([train_y, valid_y]))
save_label_map(label_map)

spec_cache: dict = {}
for n in N_MELS_VARIANTS:
    t0 = time.time()
    X_tr, y_tr = precompute_spectrograms(train_x, train_y, n, label_map)
    X_val, y_val = precompute_spectrograms(valid_x, valid_y, n, label_map)
    spec_cache[n] = (X_tr, y_tr, X_val, y_val)
    print(f'n_mels={n:3d}: train {tuple(X_tr.shape)}  val {tuple(X_val.shape)}  [{time.time()-t0:.1f}s]')

print('Precompute done.')

## Вспомогательные функции

In [ ]:
def augment_batch(x: torch.Tensor) -> torch.Tensor:
    x = x.clone()
    if random.random() < 0.5:
        x += torch.randn_like(x) * 0.02
    return x


def train_epoch(model, loader, optimizer, criterion, clip_grad: float = 1.0):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = augment_batch(x).to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.detach().argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            total_loss += criterion(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return total_loss / total, correct / total


def make_loaders(n_mels: int, batch_size: int):
    X_tr, y_tr, X_val, y_val = spec_cache[n_mels]
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size * 2, shuffle=False)
    return train_loader, valid_loader

## Optuna — поиск гиперпараметров

In [ ]:
OPTUNA_TRIALS = 40

def objective(trial):
    n_mels          = trial.suggest_categorical('n_mels',          [32, 64, 128])
    batch_size      = trial.suggest_categorical('batch_size',      [16, 32, 64])
    lr              = trial.suggest_float('lr',           1e-4, 5e-3,  log=True)
    weight_decay    = trial.suggest_float('weight_decay', 1e-5, 1e-3,  log=True)
    dropout_p1      = trial.suggest_float('dropout_p1',  0.1,  0.4)
    dropout_p2      = trial.suggest_float('dropout_p2',  0.1,  0.4)
    dropout_p3      = trial.suggest_float('dropout_p3',  0.3,  0.6)
    optimizer_name  = trial.suggest_categorical('optimizer',       ['Adam', 'AdamW'])
    num_conv_blocks = trial.suggest_categorical('num_conv_blocks', [3, 4])
    epochs          = trial.suggest_int('epochs', 30, 100)

    train_loader, valid_loader = make_loaders(n_mels, batch_size)

    model = build_model(
        num_classes=NUM_CLASSES,
        n_mels=n_mels,
        dropout_p1=dropout_p1,
        dropout_p2=dropout_p2,
        dropout_p3=dropout_p3,
        num_conv_blocks=num_conv_blocks,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    OptCls = torch.optim.AdamW if optimizer_name == 'AdamW' else torch.optim.Adam
    optimizer = OptCls(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5
    )

    best_val_acc = 0.0
    for epoch in range(epochs):
        train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = eval_epoch(model, valid_loader, criterion)
        scheduler.step(val_loss)
        best_val_acc = max(best_val_acc, val_acc)
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_acc


study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=10, interval_steps=2),
    sampler=TPESampler(n_startup_trials=10, multivariate=True, seed=42),
    study_name='radio_signal_clf',
)

print(f'Optuna: {OPTUNA_TRIALS} trials (precomputed cache)')
t0 = time.time()
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
print(f'Время поиска: {time.time()-t0:.0f}s')
print(f'Лучший val_accuracy: {study.best_value:.4f}')
print('Параметры:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## Финальное обучение с лучшими параметрами

In [ ]:
best = study.best_params
FINAL_EPOCHS = best['epochs']

train_loader, valid_loader = make_loaders(best['n_mels'], best['batch_size'])

model = build_model(
    num_classes=NUM_CLASSES,
    n_mels=best['n_mels'],
    dropout_p1=best['dropout_p1'],
    dropout_p2=best['dropout_p2'],
    dropout_p3=best['dropout_p3'],
    num_conv_blocks=best['num_conv_blocks'],
).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
OptCls = torch.optim.AdamW if best['optimizer'] == 'AdamW' else torch.optim.Adam
optimizer = OptCls(model.parameters(), lr=best['lr'], weight_decay=best['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=7, factor=0.5, min_lr=1e-6
)

print(f'Параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Финальное обучение: {FINAL_EPOCHS} эпох')

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
EARLY_STOP = 15

for epoch in range(1, FINAL_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, valid_loader, criterion)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVED_MODELS / 'model.pth')
        torch.save({
            'state_dict': model.state_dict(),
            'label_map': label_map,
            'best_params': best,
            'val_acc': val_acc,
        }, SAVED_MODELS / 'model_checkpoint.pth')
        patience_counter = 0
        marker = ' ★'
    else:
        patience_counter += 1
        marker = ''

    if epoch % 5 == 0 or epoch == 1:
        lr_cur = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{FINAL_EPOCHS} | '
              f'tr {tr_loss:.4f}/{tr_acc:.4f} | '
              f'val {val_loss:.4f}/{val_acc:.4f} | '
              f'lr={lr_cur:.2e} | {elapsed:.1f}s{marker}')

    if patience_counter >= EARLY_STOP:
        print(f'Early stop @ epoch {epoch}')
        break

print(f'\nBest val_accuracy: {best_val_acc:.4f}')

## Сохранение логов и модели

In [ ]:
log_data = {
    'best_params': best,
    'best_val_acc': best_val_acc,
    'num_classes': NUM_CLASSES,
    'label_map': label_map,
    'history': history,
}
with open(TRAINING_LOGS / 'history.json', 'w') as f:
    json.dump(log_data, f, indent=2)

print('Логи сохранены: training_logs/history.json')
print('Модель сохранена: saved_models/model.pth')

In [ ]:
import h5py

h5_path = SAVED_MODELS / 'model.h5'
state = torch.load(SAVED_MODELS / 'model.pth', map_location='cpu')

with h5py.File(h5_path, 'w') as f:
    for key, tensor in state.items():
        f.create_dataset(key, data=tensor.numpy())
    f.attrs['label_map'] = json.dumps(label_map)
    f.attrs['best_params'] = json.dumps(best)
    f.attrs['val_acc'] = best_val_acc
    f.attrs['num_classes'] = NUM_CLASSES

print(f'Модель сохранена в h5: {h5_path}')

## Графики обучения

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_acc']) + 1)

axes[0].plot(epochs_range, history['train_acc'], label='Train', color='#00b4d8')
axes[0].plot(epochs_range, history['val_acc'],   label='Val',   color='#f77f00')
axes[0].axhline(y=0.8, color='green', linestyle='--', alpha=0.5)
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, history['train_loss'], label='Train', color='#00b4d8')
axes[1].plot(epochs_range, history['val_loss'],   label='Val',   color='#f77f00')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle(f'Best val_acc={best_val_acc:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(TRAINING_LOGS / 'training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.load_state_dict(torch.load(SAVED_MODELS / 'model.pth', map_location=DEVICE))
model.eval()

all_preds, all_true = [], []
with torch.inference_mode():
    for x, y in valid_loader:
        preds = model(x.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

class_names = list(label_map.keys())
cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Val)', fontsize=13)
plt.ylabel('True')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(TRAINING_LOGS / 'confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print(classification_report(all_true, all_preds, target_names=class_names))